# Augmented Vector Analysis
## Goal: Evaluate probe performance across paraphrasers and edit scopes

This notebook analyzes the newly extracted vectors from the multi-model augmented dataset to answer:
1. How do Cohen's d and accuracy compare to the original single-paraphraser vector?
2. Do vectors generalize across all 3 paraphrasers (GPT-5, DeepSeek R1, Claude Haiku)?
3. How does performance vary by edit scope (first/mid/full)?
4. Does clipping to edited spans improve discrimination?

In [ ]:
from pathlib import Path
import sys
from collections import defaultdict
from typing import Dict, List

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from policy_vector_pipeline import (
    ActivationCollector,
    MeanDifferenceVector,
    load_dataset,
    load_model_and_tokenizer,
)

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load Dataset and Vectors

In [ ]:
# Load augmented dataset
dataset = load_dataset(PROJECT_ROOT / "data/on_policy_persona_augmented.json")

# Load both vectors
vector_full = MeanDifferenceVector.load(PROJECT_ROOT / "artifacts/qwen3_augmented_full.pt")
vector_clipped = MeanDifferenceVector.load(PROJECT_ROOT / "artifacts/qwen3_augmented_clipped.pt")

print(f"Dataset: {len(dataset)} examples")
total_on = sum(len(ex.on_policy) for ex in dataset)
total_off = sum(len(ex.off_policy) for ex in dataset)
print(f"On-policy variants: {total_on}")
print(f"Off-policy variants: {total_off}")
print(f"\nFull vector layers: {sorted(vector_full.layer_vectors.keys())}")
print(f"Clipped vector layers: {sorted(vector_clipped.layer_vectors.keys())}")

## 2. Analyze Off-Policy Distribution by Paraphraser and Edit Scope

In [ ]:
# Count variants by paraphraser and edit scope
paraphraser_counts = defaultdict(int)
scope_counts = defaultdict(int)
paraphraser_scope_counts = defaultdict(lambda: defaultdict(int))

for example in dataset:
    for variant in example.off_policy:
        meta = variant.metadata or {}
        paraphraser = meta.get("paraphraser", "unknown")
        scope = meta.get("edit_scope", "unknown")
        
        paraphraser_counts[paraphraser] += 1
        scope_counts[scope] += 1
        paraphraser_scope_counts[paraphraser][scope] += 1

print("Off-policy variants by paraphraser:")
for para, count in sorted(paraphraser_counts.items()):
    print(f"  {para}: {count}")

print("\nOff-policy variants by edit scope:")
for scope, count in sorted(scope_counts.items()):
    print(f"  {scope}: {count}")

print("\nBreakdown by paraphraser × edit scope:")
for para in sorted(paraphraser_scope_counts.keys()):
    print(f"  {para}:")
    for scope, count in sorted(paraphraser_scope_counts[para].items()):
        print(f"    {scope}: {count}")

## 3. Load Model and Collect Activations

In [ ]:
model, tokenizer = load_model_and_tokenizer(
    "Qwen/Qwen3-4B",
    device_map="auto",
    dtype="auto",
)

print(f"Model loaded on device: {model.device}")
print(f"Model dtype: {model.dtype}")

In [ ]:
# Collect activations for FULL vector (includes answer tokens)
print("Collecting activations for FULL vector evaluation...")
collector_full = ActivationCollector(
    model,
    tokenizer,
    layers=sorted(vector_full.layer_vectors.keys()),
    reduction="mean",
    response_only=True,
    include_answer=True,
)

acts_full = collector_full.collect_dataset(
    dataset,
    progress=True,
    clip_to_span=False,
)

In [ ]:
# Collect activations for CLIPPED vector (no answer, clipped to edited span)
print("\nCollecting activations for CLIPPED vector evaluation...")
collector_clipped = ActivationCollector(
    model,
    tokenizer,
    layers=sorted(vector_clipped.layer_vectors.keys()),
    reduction="mean",
    response_only=True,
    include_answer=False,
)

acts_clipped = collector_clipped.collect_dataset(
    dataset,
    progress=True,
    clip_to_span=True,
)

## 4. Compute Overall Statistics

In [ ]:
def compute_cohens_d(proj_on: torch.Tensor, proj_off: torch.Tensor) -> float:
    """Compute Cohen's d effect size."""
    mean_diff = proj_on.mean() - proj_off.mean()
    pooled_std = torch.sqrt((proj_on.var(unbiased=False) + proj_off.var(unbiased=False)) / 2)
    return (mean_diff / (pooled_std + 1e-8)).item()

def compute_accuracy(proj_on: torch.Tensor, proj_off: torch.Tensor) -> tuple[float, float, float]:
    """Compute balanced accuracy."""
    threshold = (proj_on.mean() + proj_off.mean()) / 2
    on_acc = (proj_on > threshold).float().mean().item()
    off_acc = (proj_off <= threshold).float().mean().item()
    balanced_acc = (on_acc + off_acc) / 2
    return balanced_acc, on_acc, off_acc

def evaluate_vector(acts, vector, name="Vector"):
    """Evaluate a vector across all layers."""
    results = []
    for layer, vec in sorted(vector.layer_vectors.items()):
        on_stack = torch.stack(acts["on"][layer]).to(torch.float32)
        off_stack = torch.stack(acts["off"][layer]).to(torch.float32)
        vec_norm = vec.to(torch.float32) / (torch.linalg.norm(vec.to(torch.float32)) + 1e-8)
        
        proj_on = on_stack @ vec_norm.cpu()
        proj_off = off_stack @ vec_norm.cpu()
        
        d = compute_cohens_d(proj_on, proj_off)
        bal_acc, on_acc, off_acc = compute_accuracy(proj_on, proj_off)
        
        results.append({
            "layer": layer,
            "cohens_d": d,
            "balanced_acc": bal_acc,
            "on_acc": on_acc,
            "off_acc": off_acc,
            "mean_on": proj_on.mean().item(),
            "mean_off": proj_off.mean().item(),
            "std_on": proj_on.std().item(),
            "std_off": proj_off.std().item(),
        })
    
    results.sort(key=lambda x: x["cohens_d"], reverse=True)
    
    print(f"\n{'='*60}")
    print(f"{name} - Top 5 Layers by Cohen's d")
    print(f"{'='*60}")
    for i, res in enumerate(results[:5]):
        print(f"#{i+1} Layer {res['layer']:2d}: d={res['cohens_d']:.3f}, "
              f"acc={res['balanced_acc']*100:.1f}%, "
              f"on_acc={res['on_acc']*100:.1f}%, off_acc={res['off_acc']*100:.1f}%")
    
    best = results[0]
    print(f"\nBest layer {best['layer']}: "
          f"on={best['mean_on']:.3f}±{best['std_on']:.3f}, "
          f"off={best['mean_off']:.3f}±{best['std_off']:.3f}")
    
    return results

In [ ]:
# Evaluate both vectors
results_full = evaluate_vector(acts_full, vector_full, "FULL Vector (reasoning + answer)")
results_clipped = evaluate_vector(acts_clipped, vector_clipped, "CLIPPED Vector (edited span only)")

## 5. Per-Paraphraser Analysis
**Critical Question**: Does the vector detect all paraphrasers equally well, or is it biased?

In [ ]:
def analyze_by_paraphraser(dataset, acts, vector, best_layer):
    """Analyze probe performance per paraphraser."""
    vec = vector.layer_vectors[best_layer].to(torch.float32)
    vec_norm = vec / (torch.linalg.norm(vec) + 1e-8)
    
    # Collect projections by paraphraser
    para_projections = defaultdict(list)
    
    idx = 0
    for example in dataset:
        for variant in example.off_policy:
            paraphraser = variant.metadata.get("paraphraser", "unknown")
            proj = (acts["off"][best_layer][idx].to(torch.float32) @ vec_norm.cpu()).item()
            para_projections[paraphraser].append(proj)
            idx += 1
    
    # Compute on-policy baseline
    on_stack = torch.stack(acts["on"][best_layer]).to(torch.float32)
    proj_on = on_stack @ vec_norm.cpu()
    
    print(f"\nPer-Paraphraser Analysis (Layer {best_layer}):")
    print(f"On-policy: mean={proj_on.mean():.3f}, std={proj_on.std():.3f}")
    print()
    
    para_results = []
    for para, projs in sorted(para_projections.items()):
        projs_tensor = torch.tensor(projs)
        d = compute_cohens_d(proj_on, projs_tensor)
        bal_acc, on_acc, off_acc = compute_accuracy(proj_on, projs_tensor)
        
        print(f"{para}:")
        print(f"  n={len(projs)}, mean={projs_tensor.mean():.3f}, std={projs_tensor.std():.3f}")
        print(f"  Cohen's d={d:.3f}, balanced_acc={bal_acc*100:.1f}%")
        
        para_results.append({
            "paraphraser": para,
            "n": len(projs),
            "mean": projs_tensor.mean().item(),
            "std": projs_tensor.std().item(),
            "cohens_d": d,
            "balanced_acc": bal_acc,
        })
    
    return para_results, proj_on, para_projections

In [ ]:
# Analyze FULL vector
best_layer_full = results_full[0]["layer"]
para_full, on_full, para_projs_full = analyze_by_paraphraser(
    dataset, acts_full, vector_full, best_layer_full
)

In [ ]:
# Analyze CLIPPED vector
best_layer_clipped = results_clipped[0]["layer"]
para_clipped, on_clipped, para_projs_clipped = analyze_by_paraphraser(
    dataset, acts_clipped, vector_clipped, best_layer_clipped
)

## 6. Visualize Distributions

In [ ]:
# Plot distributions by paraphraser for FULL vector
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full vector
ax = axes[0]
ax.hist(on_full.numpy(), bins=30, alpha=0.7, label="On-policy", color="blue")
colors = ["red", "green", "orange"]
for i, (para, projs) in enumerate(sorted(para_projs_full.items())):
    ax.hist(projs, bins=30, alpha=0.5, label=para.split("/")[-1], color=colors[i % len(colors)])
ax.set_xlabel("Projection onto vector")
ax.set_ylabel("Count")
ax.set_title(f"FULL Vector (Layer {best_layer_full})")
ax.legend()

# Clipped vector
ax = axes[1]
ax.hist(on_clipped.numpy(), bins=30, alpha=0.7, label="On-policy", color="blue")
for i, (para, projs) in enumerate(sorted(para_projs_clipped.items())):
    ax.hist(projs, bins=30, alpha=0.5, label=para.split("/")[-1], color=colors[i % len(colors)])
ax.set_xlabel("Projection onto vector")
ax.set_ylabel("Count")
ax.set_title(f"CLIPPED Vector (Layer {best_layer_clipped})")
ax.legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "artifacts/paraphraser_distributions.png", dpi=150)
plt.show()

## 7. Compare to Original Vector Performance

**Original vector (single paraphraser gpt-oss-20b):**
- Full reasoning: d=2.19, acc=93%
- First-line only: d=4.44, acc=97.4%

**Hypothesis:** Multi-model paraphrasing should *reduce* effect size (less fingerprinting) but improve generalization.

In [ ]:
print("\n" + "="*60)
print("COMPARISON TO ORIGINAL VECTORS")
print("="*60)

print("\nOriginal (single paraphraser):")
print("  Full reasoning: d=2.19, acc=93.0%")
print("  First-line only: d=4.44, acc=97.4%")

print("\nAugmented (3 paraphrasers):")
best_full = results_full[0]
best_clipped = results_clipped[0]
print(f"  Full reasoning: d={best_full['cohens_d']:.2f}, acc={best_full['balanced_acc']*100:.1f}%")
print(f"  Clipped to span: d={best_clipped['cohens_d']:.2f}, acc={best_clipped['balanced_acc']*100:.1f}%")

print("\nInterpretation:")
if best_full['cohens_d'] < 2.19:
    print("  ✓ Effect size DECREASED for full vector (less paraphraser fingerprinting)")
else:
    print("  ✗ Effect size INCREASED for full vector (unexpected - investigate)")

if best_clipped['cohens_d'] < 4.44:
    print("  ✓ Clipped vector has lower d than original first-line (better generalization)")
else:
    print("  ✗ Clipped vector has HIGHER d (still dominated by surface form)")

# Check cross-paraphraser consistency
para_d_values = [p['cohens_d'] for p in para_full]
if para_d_values:
    d_std = np.std(para_d_values)
    d_mean = np.mean(para_d_values)
    print(f"\nCross-paraphraser consistency (FULL vector):")
    print(f"  Mean d across paraphrasers: {d_mean:.3f}")
    print(f"  Std dev: {d_std:.3f}")
    if d_std < 0.5:
        print("  ✓ Low variance → Good generalization across paraphrasers")
    else:
        print("  ✗ High variance → Vector biased toward specific paraphraser(s)")

## 8. Save Best Layer Info for Steering Tests

In [ ]:
import json

steering_config = {
    "full_vector": {
        "path": "artifacts/qwen3_augmented_full.pt",
        "best_layer": best_layer_full,
        "cohens_d": best_full['cohens_d'],
        "balanced_acc": best_full['balanced_acc'],
    },
    "clipped_vector": {
        "path": "artifacts/qwen3_augmented_clipped.pt",
        "best_layer": best_layer_clipped,
        "cohens_d": best_clipped['cohens_d'],
        "balanced_acc": best_clipped['balanced_acc'],
    },
    "model": "Qwen/Qwen3-4B",
}

config_path = PROJECT_ROOT / "artifacts/steering_config.json"
config_path.write_text(json.dumps(steering_config, indent=2))
print(f"\nSaved steering configuration to {config_path}")
print(json.dumps(steering_config, indent=2))

## Summary

**Key Findings:**
1. **Probe performance**: Cohen's d and accuracy for both full and clipped vectors
2. **Generalization**: How consistently the vector detects all 3 paraphrasers
3. **Comparison**: Whether multi-model paraphrasing reduced paraphraser fingerprinting

**Next Steps:**
- If d is still high (>2.0): Surface form still dominates, steering likely to fail
- If d decreased substantially (<1.5): Better semantic signal, steering worth testing
- If cross-paraphraser variance is low: Good generalization, proceed to steering
- If cross-paraphraser variance is high: Vector biased, may need more diverse paraphrasing